# BLOCKED — do not run: E2 and E7 have no shared untouched test split

This draft is deliberately blocked before it reads any data. Audit evidence showed that the expansion test contains 1,183 images used by E2 training or validation, while the original E2 test contains 1,196 images used by E7 training or validation. Therefore neither split is untouched for both historic models.

The models and thresholds are fixed before the test is read:

- **E2:** MobileNetV2 at 224×224 with its earlier validation-calibrated thresholds.
- **E7:** focused MobileNetV2 at 320×320 with its earlier validation-calibrated thresholds.

Before running, attach these private Kaggle datasets:

1. `GOV-01 V2 Multilabel N-RDD2024 Expansion`
2. A small private dataset containing the two output ZIPs for E2 and E7.

Do not attach data or run this notebook. A valid final-evaluation protocol must be chosen before a replacement notebook is created.

In [ ]:
# Cell 1 — safety block. No dataset, model, or protected-test file is read.
raise RuntimeError(
    'STOP: This historic E2-versus-E7 comparison is invalid because neither available test split is untouched for both models. '
    'No protected-test data has been loaded by this notebook.'
)

# The remaining draft cells are intentionally unreachable until a valid protocol is approved.
from pathlib import Path
import csv
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support

CONDITIONS = ('crack', 'pothole', 'repaired_road', 'manhole_cover', 'unpaved_road', 'road_marking', 'speed_bump')
EXPECTED_TEST_IMAGES = 3978
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/gov01_final_e2_e7_test')
OUTPUT_ROOT = WORK_ROOT / 'evidence'
SESSION_MARKER = WORK_ROOT / 'protected_test_evaluated_this_session.json'

# These values are deliberately written here, before the protected test is read. Do not edit them after evaluation.
LOCKED_MODELS = {
    'E2': {
        'model_filename': 'v2_multilabel_e2_finetuned_best.keras',
        'image_size': (224, 224),
        'thresholds': {'crack': 0.55, 'pothole': 0.30, 'repaired_road': 0.30, 'manhole_cover': 0.35, 'unpaved_road': 0.55, 'road_marking': 0.75, 'speed_bump': 0.80},
    },
    'E7': {
        'model_filename': 'e7_focused_best.keras',
        'image_size': (320, 320),
        'thresholds': {'crack': 0.46, 'pothole': 0.46, 'repaired_road': 0.56, 'manhole_cover': 0.49, 'unpaved_road': 0.71, 'road_marking': 0.50, 'speed_bump': 0.50},
    },
}

if SESSION_MARKER.exists():
    raise RuntimeError('This Kaggle session has already evaluated the protected test. Do not run it again.')

def exactly_one_file(filename):
    matches = [path for path in INPUT_ROOT.rglob(filename) if path.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one {filename!r} under /kaggle/input; found {matches}')
    return matches[0]

def safe_extract(archive, members, destination):
    destination = destination.resolve()
    for member in members:
        target = (destination / member.filename).resolve()
        if target != destination and destination not in target.parents:
            raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
    archive.extractall(destination, members=members)

def find_test_root():
    labels = [path for path in INPUT_ROOT.rglob('labels.csv') if path.parent.name == 'test']
    if len(labels) == 1:
        return labels[0].parent.parent
    if len(labels) > 1:
        raise RuntimeError(f'Ambiguous protected-test labels found: {labels}')

    # Kaggle normally unpacks uploaded ZIPs. This fallback extracts only the protected test if it did not.
    for archive_path in INPUT_ROOT.rglob('*.zip'):
        with zipfile.ZipFile(archive_path) as archive:
            members = [member for member in archive.infolist() if member.filename.endswith('/test/labels.csv') or '/test/images/' in member.filename]
            label_members = [member for member in members if member.filename.endswith('/test/labels.csv')]
            if len(label_members) == 1:
                safe_extract(archive, members, WORK_ROOT)
                return WORK_ROOT / Path(label_members[0].filename).parent.parent
    raise FileNotFoundError('Could not find the protected test labels. Attach the N-RDD expansion dataset before running.')

E2_MODEL_PATH = exactly_one_file(LOCKED_MODELS['E2']['model_filename'])
E7_MODEL_PATH = exactly_one_file(LOCKED_MODELS['E7']['model_filename'])
TEST_ROOT = find_test_root()
TEST_LABELS_PATH = TEST_ROOT / 'test' / 'labels.csv'

print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('E2 locked model:', E2_MODEL_PATH)
print('E7 locked model:', E7_MODEL_PATH)
print('Protected test root:', TEST_ROOT)
print('Training and validation data are not loaded by this notebook.')

In [ ]:
# Cell 2 — verify and read only the protected test labels.
def read_protected_test():
    paths, labels, masks = [], [], []
    with TEST_LABELS_PATH.open(newline='', encoding='utf-8') as source:
        for row in csv.DictReader(source):
            image_path = TEST_ROOT / 'test' / Path(row['materialized_image'].replace(chr(92), '/'))
            if not image_path.is_file():
                raise FileNotFoundError(f'Missing protected-test image: {image_path}')
            label_row, mask_row = [], []
            for condition in CONDITIONS:
                known = int(row[f'{condition}_known'])
                value = row[f'{condition}_present']
                if known not in (0, 1) or (known == 0 and value != '') or (known == 1 and value not in ('0', '1')):
                    raise ValueError(f'Invalid protected-test label: {row["record_id"]} / {condition}')
                label_row.append(float(value) if known else 0.0)
                mask_row.append(float(known))
            paths.append(str(image_path))
            labels.append(label_row)
            masks.append(mask_row)
    return np.asarray(paths), np.asarray(labels, dtype='float32'), np.asarray(masks, dtype='float32')

test_paths, test_labels, test_masks = read_protected_test()
if len(test_paths) != EXPECTED_TEST_IMAGES:
    raise RuntimeError(f'Protected test has {len(test_paths)} images; expected {EXPECTED_TEST_IMAGES}. Stop and inspect the attached dataset.')

print(f'Protected test verified: {len(test_paths)} images')
for index, condition in enumerate(CONDITIONS):
    known = int(test_masks[:, index].sum())
    positives = int((test_labels[:, index] * test_masks[:, index]).sum())
    print(f'{condition:16} known={known:5} positive={positives:5}')
print('Unknown labels remain excluded from every score.')

In [ ]:
# Cell 3 — one-time fixed-threshold prediction and scoring. This is not training or calibration.
def make_image_dataset(paths, image_size, batch_size=16):
    def load_image(path):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, image_size)
        return tf.cast(image, tf.float32)
    return tf.data.Dataset.from_tensor_slices(paths).map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

def score_locked_model(model_name, model_path, settings):
    model = tf.keras.models.load_model(model_path, compile=False)
    probabilities = model.predict(make_image_dataset(test_paths, settings['image_size']), verbose=1)
    if probabilities.shape != test_labels.shape:
        raise RuntimeError(f'{model_name} prediction shape {probabilities.shape} does not match expected {test_labels.shape}')

    predicted = np.zeros_like(test_labels, dtype='int32')
    per_condition, matrices = {}, {}
    f1_values = []
    correct_known = 0
    known_total = 0
    for index, condition in enumerate(CONDITIONS):
        threshold = settings['thresholds'][condition]
        known = test_masks[:, index].astype(bool)
        actual = test_labels[known, index].astype(int)
        condition_predictions = (probabilities[known, index] >= threshold).astype(int)
        predicted[known, index] = condition_predictions
        precision, recall, f1, support = precision_recall_fscore_support(actual, condition_predictions, average='binary', zero_division=0)
        matrix = confusion_matrix(actual, condition_predictions, labels=[0, 1])
        true_negative, false_positive, false_negative, true_positive = matrix.ravel()
        per_condition[condition] = {
            'threshold_locked_before_test': float(threshold),
            'known_images': int(len(actual)), 'positive_images': int(actual.sum()),
            'precision': float(precision), 'recall': float(recall), 'f1': float(f1),
            'true_negative': int(true_negative), 'false_positive': int(false_positive),
            'false_negative': int(false_negative), 'true_positive': int(true_positive),
        }
        matrices[condition] = matrix
        f1_values.append(float(f1))
        correct_known += int((actual == condition_predictions).sum())
        known_total += int(len(actual))

    result = {
        'model': model_name, 'model_file': model_path.name, 'image_size': list(settings['image_size']),
        'selection_data': 'protected test only', 'training_or_tuning_performed': False,
        'protected_test_images': int(len(test_paths)),
        'masked_macro_f1': float(np.mean(f1_values)),
        'masked_label_accuracy': float(correct_known / known_total),
        'per_condition': per_condition,
    }
    return result, matrices

if SESSION_MARKER.exists():
    raise RuntimeError('Protected test was already evaluated in this Kaggle session. Do not rerun it.')

e2_result, e2_matrices = score_locked_model('E2', E2_MODEL_PATH, LOCKED_MODELS['E2'])
e7_result, e7_matrices = score_locked_model('E7', E7_MODEL_PATH, LOCKED_MODELS['E7'])

SESSION_MARKER.parent.mkdir(parents=True, exist_ok=True)
SESSION_MARKER.write_text(json.dumps({'completed': True, 'models': ['E2', 'E7']}, indent=2), encoding='utf-8')
print('E2 masked macro F1:', f"{e2_result['masked_macro_f1']:.4f}")
print('E7 masked macro F1:', f"{e7_result['masked_macro_f1']:.4f}")
print('One-time session marker written. Do not rerun this cell.')

In [ ]:
# Cell 4 — save final evidence. This cell never changes a model, threshold, or score.
if 'e2_result' not in globals() or 'e7_result' not in globals():
    raise RuntimeError('Run the one-time scoring cell exactly once before saving evidence.')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
final_report = {
    'experiment': 'GOV-01 V2 multi-label final protected-test E2 versus E7 comparison',
    'selection_data_before_test': 'E2 and E7 models plus thresholds were locked from earlier validation work',
    'test_split': 'v2_multilabel_n_rdd_expansion/protected test',
    'protected_test_reused_for_tuning': False,
    'training_or_tuning_performed': False,
    'models': {'E2': e2_result, 'E7': e7_result},
}
(OUTPUT_ROOT / 'final_e2_vs_e7_protected_test_metrics.json').write_text(json.dumps(final_report, indent=2) + '\n', encoding='utf-8')

rows = []
for result in (e2_result, e7_result):
    for condition, values in result['per_condition'].items():
        rows.append({'model': result['model'], 'condition': condition, **values})
pd.DataFrame(rows).to_csv(OUTPUT_ROOT / 'final_e2_vs_e7_per_condition.csv', index=False)

figure, axes = plt.subplots(2, len(CONDITIONS), figsize=(21, 6))
for row_index, (model_name, matrices) in enumerate((('E2', e2_matrices), ('E7', e7_matrices))):
    for column_index, condition in enumerate(CONDITIONS):
        axis = axes[row_index, column_index]
        matrix = matrices[condition]
        axis.imshow(matrix, cmap='Blues')
        axis.set_title(f'{model_name}: {condition}', fontsize=9)
        axis.set_xticks([0, 1], ['no', 'yes'])
        axis.set_yticks([0, 1], ['no', 'yes'])
        if column_index == 0:
            axis.set_ylabel('actual')
        if row_index == 1:
            axis.set_xlabel('predicted')
        cut = matrix.max() / 2 if matrix.max() else 0
        for y in range(2):
            for x in range(2):
                axis.text(x, y, str(matrix[y, x]), ha='center', va='center', fontsize=9, color='white' if matrix[y, x] > cut else 'black')
figure.suptitle('Final protected-test confusion matrices — fixed thresholds', y=1.02)
figure.tight_layout()
figure.savefig(OUTPUT_ROOT / 'final_e2_vs_e7_confusion_matrices.png', dpi=160, bbox_inches='tight')
plt.show()

comparison = pd.DataFrame([
    {'model': 'E2', 'masked_macro_f1': e2_result['masked_macro_f1'], 'masked_label_accuracy': e2_result['masked_label_accuracy']},
    {'model': 'E7', 'masked_macro_f1': e7_result['masked_macro_f1'], 'masked_label_accuracy': e7_result['masked_label_accuracy']},
])
print(comparison.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
print('Evidence folder:', OUTPUT_ROOT)

In [ ]:
# Cell 5 — package the final evidence for download. Keep this ZIP outside Git.
required = [
    OUTPUT_ROOT / 'final_e2_vs_e7_protected_test_metrics.json',
    OUTPUT_ROOT / 'final_e2_vs_e7_per_condition.csv',
    OUTPUT_ROOT / 'final_e2_vs_e7_confusion_matrices.png',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing final evidence: {missing}')
result_zip = Path('/kaggle/working/v2_multilabel_final_e2_vs_e7_protected_test_output.zip')
if result_zip.exists():
    result_zip.unlink()
shutil.make_archive(str(result_zip.with_suffix('')), 'zip', OUTPUT_ROOT)
print('Completed final evidence ZIP:', result_zip)
print('Download it from Kaggle Output. Do not rerun the protected-test scoring cell or use this result for more tuning.')